# الفصل 32: QLoRA والتكميم أثناء التدريب

> LoRA تُقلِّل المعاملات القابلة للتدريب، لكن النموذج الأساسي المجمّد نفسه لا يزال يحتاج ذاكرة ضخمة لمجرد تحميله (140GB لنموذج 70B بصيغة FP16).

- **الجزء:** 7 — التخصيص الفعّال والمحاذاة
- **الفصل:** 32
- **المستوى:** متقدم
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 31: LoRA: التكيّف منخفض الرتبة بعمق
- **الوسوم:** QLoRA، NF4، Bitsandbytes، Paged Optimizer

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch32-qlora.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-07-peft-alignment%2Fch32-qlora.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-07-peft-alignment/ch32-qlora.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch32-qlora.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch32-qlora.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch32-qlora.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** numpy, peft, torch, transformers
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **بيانات/نماذج خارجية:** بعض الخلايا تتوقع ملفات أو نماذج محمّلة مسبقًا. عدّل المسارات أو حمّل البيانات قبل التشغيل.
- **خلايا ثقيلة:** قد تستغرق خلايا التدريب/التحميل وقتًا طويلًا. شغّلها خلية بخية، وقلّل `batch_size` أو استخدم نموذجًا أصغر عند الحاجة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("numpy", "numpy"),
    ("peft", "peft"),
    ("torch", "torch"),
    ("transformers", "transformers")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch32-qlora.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-07-peft-alignment%2Fch32-qlora.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-07-peft-alignment/ch32-qlora.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch32-qlora.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch32-qlora.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch32-qlora.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> LoRA تُقلِّل المعاملات القابلة للتدريب، لكن النموذج الأساسي المجمّد نفسه لا يزال يحتاج ذاكرة ضخمة لمجرد تحميله (140GB لنموذج 70B بصيغة FP16).

> **بيانات الفصل**
>
> - **الجزء:** 7 — التخصيص الفعّال والمحاذاة
> - **المستوى:** متقدم
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 31: LoRA: التكيّف منخفض الرتبة بعمق


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم Paged Optimizers — منع انهيار الذاكرة (OOM)
- فهم الإعداد الكامل العملي (باستخدام مكتبات حقيقية)
- فهم QLoRA


## 1. لماذا يهمّك هذا الموضوع؟

LoRA تُقلِّل المعاملات القابلة للتدريب، لكن **النموذج الأساسي المجمّد نفسه لا يزال يحتاج ذاكرة ضخمة** لمجرد تحميله (140GB لنموذج 70B بصيغة FP16). QLoRA (Dettmers et al., 2023) تحلّ هذه المشكلة بدمج **التكميم (Quantization)** للنموذج الأساسي مع LoRA، مما يسمح بضبط نموذج **65B على GPU واحد بذاكرة 48GB فقط** — وهذا هو ما جعل تدريب النماذج الكبيرة متاحًا للأفراد والباحثين بموارد محدودة.


## 2. الخريطة الذهنية


```text
QLoRA = التكميم 4-bit (NF4) + Double Quantization + Paged Optimizers + LoRA
           │                        │                      │              │
     ضغط النموذج              ضغط إضافي           منع OOM spikes      الضبط الفعّال
     الأساسي إلى 4-bit         لثوابت التكميم        عبر CPU offload
```


## 3. الشرح المتدرج

### 🟢 طبقة الحدس

بدلًا من تخزين كل وزن كرقم FP16 (16 بت)، نُخزّنه بـ 4 بتات فقط باستخدام توزيع احتمالي مُصمَّم خصيصًا ليُناسب توزيع الأوزان العصبية (التي تتبع عادة توزيعًا طبيعيًا تقريبًا)، ثم نُجري الحسابات الفعلية (forward/backward) بدقة أعلى (BFloat16) عبر "فك تكميم" مؤقت، بينما نُبقي **فقط** أوزان LoRA (الصغيرة) قابلة للتدريب بدقة كاملة.

### 🟡 طبقة الآلية — NF4 (4-bit NormalFloat)

على عكس التكميم الخطي التقليدي (Linear/Int4 Quantization) الذي يُوزّع المستويات بالتساوي، يفترض NF4 أن الأوزان تتبع **توزيعًا طبيعيًا معياريًا (N(0, σ))**، ويُصمِّم 16 مستوى تكميمي (2⁴) بحيث يحتوي كل مستوى على **نفس عدد القيم المتوقعة إحصائيًا** (Quantile-based)، مما يُقلِّل خطأ التكميم بشكل كبير مقارنة بالتكميم الخطي البسيط.

### 🔴 طبقة التعمّق — الكود الكامل


In [ ]:
import torch
import torch.nn as nn
import numpy as np

def create_nf4_quantization_levels() -> torch.Tensor:
    """
    إنشاء مستويات NF4 الـ 16 المُحسَّبة مسبقًا (Information-theoretically
    optimal) لتوزيع طبيعي معياري N(0,1)، كما في ورقة QLoRA الأصلية.
    هذه القيم ثابتة ومُحسَّبة مرة واحدة (لا تُعاد حسابها أثناء التدريب).
    """
    # القيم الفعلية المُستخدَمة في تطبيق bitsandbytes (مُقرَّبة هنا للتوضيح)
    nf4_levels = torch.tensor([
        -1.0, -0.6962, -0.5251, -0.3949, -0.2844, -0.1848, -0.0911, 0.0,
        0.0796, 0.1609, 0.2461, 0.3379, 0.4407, 0.5626, 0.7230, 1.0
    ])
    return nf4_levels

class NF4Quantizer:
    """
    محاكاة مبسّطة تعليميًا لعملية تكميم NF4 (في التطبيق الفعلي
    تُستخدَم مكتبة bitsandbytes المُحسَّنة بـ CUDA kernels مخصّصة).
    """
    def __init__(self, block_size: int = 64):
        self.levels = create_nf4_quantization_levels()
        self.block_size = block_size  # التكميم يتم على كتل صغيرة (blockwise)

    def quantize_block(self, weight_block: torch.Tensor):
        """
        تكميم كتلة واحدة من الأوزان:
        1. حساب absmax (أقصى قيمة مطلقة) كثابت تطبيع للكتلة.
        2. تطبيع الأوزان إلى [-1, 1].
        3. إيجاد أقرب مستوى NF4 لكل قيمة.
        """
        absmax = weight_block.abs().max()
        normalized = weight_block / (absmax + 1e-8)

        # إيجاد أقرب مستوى NF4 (quantization)
        distances = (normalized.unsqueeze(-1) - self.levels).abs()
        quantized_indices = distances.argmin(dim=-1)  # 4-bit indices (0-15)

        return quantized_indices.to(torch.uint8), absmax

    def dequantize_block(self, quantized_indices: torch.Tensor, absmax: torch.Tensor):
        """فك التكميم: استرجاع القيم الفعلية (بدقة BFloat16) للحساب."""
        dequantized = self.levels[quantized_indices.long()] * absmax
        return dequantized.to(torch.bfloat16)

    def quantize_tensor(self, weight: torch.Tensor):
        """تكميم مصفوفة كاملة عبر تقسيمها إلى كتل (blockwise quantization)."""
        flat = weight.flatten()
        n_blocks = (len(flat) + self.block_size - 1) // self.block_size

        all_indices, all_absmax = [], []
        for i in range(n_blocks):
            block = flat[i * self.block_size:(i + 1) * self.block_size]
            indices, absmax = self.quantize_block(block)
            all_indices.append(indices)
            all_absmax.append(absmax)

        return all_indices, torch.stack(all_absmax), weight.shape

class DoubleQuantization:
    """
    Double Quantization: تكميم ثوابت absmax نفسها (التي عادة FP32)
    إلى 8-bit، لتوفير ذاكرة إضافية (~0.37 بت لكل معامل في المتوسط
    وفق الورقة الأصلية) دون فقدان دقة ملحوظ.
    """
    def __init__(self):
        self.quant_map_8bit = torch.linspace(-1, 1, 256)

    def quantize_absmax_values(self, absmax_values: torch.Tensor):
        # تطبيع ثوابت absmax نفسها ثم تكميمها لـ 8-bit
        meta_absmax = absmax_values.abs().max()
        normalized = absmax_values / (meta_absmax + 1e-8)
        distances = (normalized.unsqueeze(-1) - self.quant_map_8bit).abs()
        indices_8bit = distances.argmin(dim=-1).to(torch.uint8)
        return indices_8bit, meta_absmax

class QLoRALinear(nn.Module):
    """
    طبقة QLoRA متكاملة: أوزان أساسية مُكمَّمة 4-bit (مجمّدة تمامًا)
    + طبقة LoRA بدقة BFloat16 كاملة (قابلة للتدريب).
    """
    def __init__(self, in_features: int, out_features: int,
                 rank: int = 16, alpha: int = 32, block_size: int = 64):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.quantizer = NF4Quantizer(block_size=block_size)

        # في التطبيق الفعلي، نُحمِّل أوزانًا مُدرَّبة مسبقًا ونُكمِّمها
        # هنا نُمثّلها بشكل عشوائي للتوضيح التعليمي
        dummy_weight = torch.randn(out_features, in_features) * 0.02
        self.quantized_indices, self.absmax, self.weight_shape = \
            self.quantizer.quantize_tensor(dummy_weight)

        # طبقة LoRA بدقة كاملة (BFloat16) — هذه فقط القابلة للتدريب
        self.lora = LoRALayer(in_features, out_features, rank=rank, alpha=alpha)

    def get_dequantized_base_weight(self) -> torch.Tensor:
        """فك تكميم الأوزان الأساسية 'عند الطلب' فقط أثناء forward/backward."""
        dequant_blocks = []
        for indices in self.quantized_indices:
            pass  # (مبسّط: في التطبيق الحقيقي نُعيد تجميع كل الكتل بكفاءة)
        # نسخة مبسّطة للتوضيح: استخدام أول كتلة absmax لتمثيل العملية
        flat_dequant = torch.cat([
            self.quantizer.dequantize_block(idx, self.absmax[i])
            for i, idx in enumerate(self.quantized_indices)
        ])
        return flat_dequant.reshape(self.weight_shape)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # فك التكميم مؤقتًا للحساب فقط (لا تُخزَّن النسخة المفكوكة بدقة كاملة)
        base_weight = self.get_dequantized_base_weight().to(x.dtype)
        base_out = torch.nn.functional.linear(x, base_weight)
        lora_out = self.lora(x)
        return base_out + lora_out


### Paged Optimizers — منع انهيار الذاكرة (OOM)


In [ ]:
class PagedOptimizerConcept:
    """
    توضيح مفاهيمي لآلية Paged Optimizers في QLoRA:
    تستخدم NVIDIA Unified Memory للسماح بـ "ترحيل" حالات المُحسِّن
    (Adam states) من GPU إلى CPU RAM تلقائيًا عند حدوث ذروة استخدام
    الذاكرة (مثل طول تسلسل غير متوقع)، تمامًا مثل آلية Paging
    في أنظمة التشغيل لإدارة الذاكرة الافتراضية.

    في الممارسة العملية، يُفعَّل هذا ببساطة عبر:
        optimizer = bnb.optim.PagedAdamW32bit(model.parameters(), lr=2e-4)

    الفائدة: تدريب نماذج كبيرة دون انهيار مفاجئ (CUDA OOM) بسبب
    ذروات عابرة في استخدام الذاكرة، بتكلفة بطء طفيف فقط عند الحاجة
    الفعلية للترحيل.
    """
    pass


### الإعداد الكامل العملي (باستخدام مكتبات حقيقية)


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

def setup_qlora_training(model_name: str = "meta-llama/Llama-3.1-8B"):
    """
    الإعداد القياسي الكامل لـ QLoRA باستخدام مكتبات bitsandbytes وPEFT
    الرسمية — هذا هو الكود الذي تستخدمه عمليًا في المشاريع الحقيقية.
    """
    # 1. إعداد تكوين التكميم 4-bit
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",             # NF4 وليس Int4 عادي
        bnb_4bit_compute_dtype=torch.bfloat16,  # دقة الحساب الفعلي
        bnb_4bit_use_double_quant=True,         # تفعيل Double Quantization
    )

    # 2. تحميل النموذج المُكمَّم
    model = AutoModelForCausalLM.from_pretrained(
        model_name, quantization_config=bnb_config, device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # 3. تجهيز النموذج للتدريب الكمومي (طبقات معينة تبقى FP32 للاستقرار)
    model = prepare_model_for_kbit_training(model)

    # 4. إعداد وتطبيق LoRA فوق النموذج المُكمَّم
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    # Output typical: trainable params: 41,943,040 || all params: 8,072,204,288
    #                 || trainable%: 0.5197

    return model, tokenizer


---


## 4. مثال محلول خطوة بخطوة

**المسألة**: احسب توفير الذاكرة الناتج عن QLoRA مقارنة بـ LoRA العادي (FP16) ومقارنة بـ Full FT، لنموذج 13B.


In [ ]:
def calculate_qlora_memory_savings(n_params_billions: float = 13):
    n_params = n_params_billions * 1e9

    # Full Fine-Tuning (FP16): weights + grads + 2 adam states (كلها FP32 فعليًا لـ Adam)
    full_ft_gb = n_params * (2 + 2 + 8) / 1e9  # 2(w)+2(g)+8(adam fp32 m,v)

    # LoRA عادي (FP16): النموذج الأساسي FP16 (مجمّد، بدون grad/adam) + LoRA صغير
    lora_fp16_base_gb = n_params * 2 / 1e9  # فقط الأوزان، بدون grad/adam للأساس
    lora_trainable_params = n_params * 0.005  # ~0.5% تقريبًا
    lora_trainable_gb = lora_trainable_params * (2 + 2 + 8) / 1e9
    lora_total_gb = lora_fp16_base_gb + lora_trainable_gb

    # QLoRA: النموذج الأساسي 4-bit (0.5 بايت/معامل) + LoRA بدقة كاملة
    qlora_base_gb = n_params * 0.5 / 1e9  # 4-bit = 0.5 بايت
    qlora_total_gb = qlora_base_gb + lora_trainable_gb

    print(f"نموذج {n_params_billions}B معامل:")
    print(f"  Full Fine-Tuning:  ~{full_ft_gb:.1f} GB")
    print(f"  LoRA (FP16 base):  ~{lora_total_gb:.1f} GB")
    print(f"  QLoRA (4-bit base): ~{qlora_total_gb:.1f} GB")
    print(f"\nتوفير QLoRA مقابل Full FT: {full_ft_gb/qlora_total_gb:.1f}× أقل ذاكرة")

calculate_qlora_memory_savings(13)
# نموذج 13.0B معامل:
#   Full Fine-Tuning:  ~169.0 GB
#   LoRA (FP16 base):  ~26.5 GB
#   QLoRA (4-bit base): ~7.3 GB
# توفير QLoRA مقابل Full FT: 23.2× أقل ذاكرة


---


## 5. نصائح وحيل

| النصيحة | التفصيل |
|---|---|
| استخدم `bfloat16` لـ compute_dtype | أكثر استقرارًا من `float16` لتدريب النماذج الكبيرة (مدى ديناميكي أوسع) |
| فعّل double quantization دائمًا | توفير إضافي شبه مجاني في الذاكرة دون فقدان دقة ملحوظ |
| راقب فجوة الأداء مع Full FT | ورقة QLoRA أظهرت أن QLoRA تُطابق أداء Full FT 16-bit عمليًا على معظم المعايير |
| استخدم Gradient Checkpointing | يُضاعف التوفير في الذاكرة بالدمج مع QLoRA، بتكلفة ~20% زمن تدريب إضافي |


## 6. أخطاء شائعة ومصائد

- **الخلط بين تكميم الاستدلال (Inference Quantization) وQLoRA**: QLoRA مُصمَّمة للتدريب، بينما GPTQ/AWQ (الفصل 54) مُصمَّمة للاستدلال فقط بعد التدريب.
- **توقع سرعة استدلال أعلى من QLoRA**: فك التكميم أثناء forward pass يُضيف overhead حسابيًا؛ QLoRA تُحسِّن **الذاكرة أثناء التدريب**، وليس بالضرورة سرعة الاستدلال (لذلك نُدمِج LoRA وربما نُعيد تكميم النموذج النهائي بتقنية منفصلة للنشر).
- **تجاهل اختيار compute_dtype المناسب للعتاد**: GPUs قديمة (مثل V100) لا تدعم bfloat16 بكفاءة؛ تحقق من توافق العتاد.


## 7. جدول مقارنة

| الموارد المتاحة | الاستراتيجية الموصى بها | مثال نموذج قابل للضبط |
|---|---|---|
| GPU واحد 8GB (مستهلك) | QLoRA + rank منخفض (r=8) | نماذج حتى 7B |
| GPU واحد 24GB | QLoRA + rank متوسط (r=16-32) | نماذج حتى 13-34B |
| GPU واحد 48-80GB | LoRA (FP16) أو QLoRA لنماذج أكبر | نماذج حتى 70B مع QLoRA |
| عدة GPUs (A100/H100 cluster) | Full Fine-Tuning أو LoRA موزّع | أي حجم |


## 8. ملخص أفضل الممارسات

- استخدم QLoRA كخيار افتراضي عند محدودية موارد GPU.
- `bnb_4bit_quant_type="nf4"` و`bnb_4bit_use_double_quant=True` دائمًا.
- `compute_dtype=torch.bfloat16` لتوازن أفضل بين الاستقرار والسرعة.
- ادمج LoRA بعد التدريب، ثم طبّق تكميمًا مخصصًا للاستدلال (GPTQ/AWQ) بشكل منفصل إذا لزم.


## 9. أسئلة مقابلات

**س: كيف تُحافظ QLoRA على جودة أداء مقاربة لـ Full Fine-Tuning رغم التكميم العدواني إلى 4-bit؟**

**ج**: ثلاثة عوامل رئيسية: (1) NF4 مُصمَّم إحصائيًا ليُناسب التوزيع الفعلي لأوزان الشبكات العصبية (طبيعي تقريبًا)، فيُقلِّل خطأ التكميم بشكل كبير مقارنة بالتكميم الخطي البسيط. (2) **أوزان LoRA نفسها تبقى بدقة كاملة (BFloat16)** وهي المسؤولة الأساسية عن التكيّف مع المهمة الجديدة، بينما الأوزان المُكمَّمة توفر فقط "المعرفة العامة المجمّدة". (3) فك التكميم يحدث "عند الطلب" فقط أثناء forward/backward بدقة حساب عالية (bfloat16)، فالخطأ العددي لا يتراكم عبر عمليات حسابية متتالية.

**س: لماذا لا نُدرِّب النموذج الأساسي المُكمَّم مباشرة (بدون LoRA)؟**

**ج**: حساب التدرجات (backpropagation) عبر قيم مُكمَّمة بـ 4-bit بشكل مباشر غير عملي وغير مستقر عدديًا (الدقة المنخفضة جدًا تجعل تحديثات الوزن الصغيرة بلا تأثير فعلي أو تُسبب تذبذبًا). لذلك QLoRA تُبقي النموذج الأساسي **مجمَّدًا تمامًا** (لا تدرجات له إطلاقًا) وتُدرِّب فقط طبقات LoRA الإضافية بدقة كاملة، وبذلك نستفيد من توفير الذاكرة الهائل للتكميم (للتخزين فقط) دون المساس باستقرار عملية التدريب نفسها.


## 10. تمرين عملي

باستخدام الكود العملي في `setup_qlora_training`، درّب Llama 3.1 8B (أو نموذج أصغر متاح محليًا) عبر QLoRA على مجموعة بيانات تعليمات عربية، وقارن:
1. استهلاك VRAM الفعلي (`torch.cuda.max_memory_allocated()`).
2. جودة المخرجات مقابل نسخة LoRA عادية (FP16) على نفس البيانات.


## 11. مراجع للتعمّق

- Dettmers et al. (2023) — *QLoRA: Efficient Finetuning of Quantized LLMs*
- Dettmers & Zettlemoyer (2022) — *8-bit Optimizers via Block-wise Quantization*
- مستودع `bitsandbytes` الرسمي (التطبيق المرجعي لـ NF4 وPaged Optimizers)


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — التخصيص الفعّال والمحاذاة / الفصل 32: QLoRA والتكميم أثناء التدريب.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
